# PilotFormer-R: one Transformer for OFDM channel estimation under arbitrary and unseen pilot layouts
### Layout-agnostic set-to-grid attention with a robust-LMMSE filter-bank prior, for doubly selective 3GPP channels

**Problem.** Most deep-learning OFDM channel estimators (ChannelNet, ReEsNet, Channelformer, ...) are trained for one pilot
layout; a different pilot density or reference-signal pattern requires retraining.

**Method.** The received pilots form an unordered set of tokens that carry their own time-frequency coordinates. A Transformer
encoder processes the set, and every resource element (RE) is a coordinate query that cross-attends to it with a learnable
locality bias. A bank of robust LMMSE filters (Li-Cimini-Sollenberger, designed for 100 / 400 / 1000 Hz) is computed in closed
form for any layout, and the network outputs a per-RE mixture of the bank estimates plus a residual, conditioned on the SNR.

**What this notebook does**
1. Trains PilotFormer-R (3 seeds), its ablations (Lite, no filter bank, single-layout training) and the CNN-R control.
2. Trains two published baselines on the same data, loss and budget:
   * **ChannelNet** (Soltani et al., IEEE Commun. Lett. 2019): SRCNN (64@9x9, 32@1x1, 5x5) + 20-layer DnCNN on the interpolated pilot grid;
   * **Channelformer** (Luan and Thompson, IEEE TWC 2023): attention encoder on the pilot LS vector, residual-conv decoder and
     fully connected upsampling. Its fully connected layers fix the number of pilots, so it is trained and tested on NR-DMRS only.
3. Evaluates all estimators (plus LS, LMMSE and oracle LMMSE) on EVA, ETU and 3GPP TR 38.901 TDL-A / TDL-C channels, four pilot
   layouts, 0 to 30 dB SNR, a Doppler sweep, SNR mismatch, BER and complexity, and writes every result to CSV.

**Running on Kaggle:** Accelerator: **GPU T4**, then **Save Version > Save & Run All (Commit)**. A full run takes 6 to 7 hours.
Optional: attach the output of an earlier run (*Add Input*) and its checkpoints are loaded instead of retrained.
Quick trial: set the environment variables `SEEDS=0` and `STEPS=3000`, or edit `CFG`.

In [ ]:
import os, time, json, math, glob, zipfile, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import cKDTree
warnings.filterwarnings("ignore")

CFG = dict(
    out_dir     = os.environ.get("OUT_DIR", "/kaggle/working"),
    n_sc        = 72,            # sub-carriers (6 PRBs)
    n_sym       = 14,            # OFDM symbols per slot
    scs         = 15e3,          # sub-carrier spacing [Hz]
    train_steps = int(os.environ.get("STEPS", 20000)),
    batch       = int(os.environ.get("BATCH", 128)),
    lr          = 5e-4,
    d_model     = 128,
    dec_mlp     = 2,             # decoder MLP expansion; the decoder runs on all K*L REs, so it dominates FLOPs
    train_snr   = (0.0, 30.0),   # dB, uniform
    train_fd    = (0.0, 1000.0), # Hz, uniform  (1200 Hz = beyond training)
    bank_fd     = [100.0, 400.0, 1000.0],   # Doppler design points of the robust-LMMSE filter bank
    train_prof  = ["EPA", "EVA"],# ETU is held out (profile shape never seen)
    train_dscale= (0.5, 2.2),    # delay-spread scaling of the training profiles (EVA x2.2 -> 5.5 us max excess delay)
    test_snrs   = list(range(0, 31, 5)),
    n_test      = int(os.environ.get("NTEST", 1000)),   # slots per (condition, SNR)
    n_cov       = int(os.environ.get("NCOV", 5000)),          # channels for LMMSE covariance estimates
    seed        = 0,
    seeds       = [int(x) for x in os.environ.get("SEEDS", "0,1,2").split(",")],   # PilotFormer & CNN; ablations use seeds[0]
    ablations   = os.environ.get("ABLATIONS", "1") == "1",
    abl_no_ds   = os.environ.get("ABL_DS", "0") == "1",    # extra ablation: PilotFormer trained without delay-spread scaling
    snr_offsets = [-3.0, -1.5, 0.0, 1.5, 3.0],              # dB error of the SNR estimate given to SNR-aware methods
    ckpt_search = os.environ.get("CKPT_DIR", "/kaggle/input"), # where to look for checkpoints of a previous run
    base_seeds  = [int(x) for x in os.environ.get("BASE_SEEDS", "0").split(",")],   # seeds for the published baselines
    tdl_ds_ns   = 300.0,                                     # TR 38.901 TDL delay spread ("long", Table 7.7.3-1)
)
os.makedirs(CFG["out_dir"], exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEV.type == "cuda"
K, L = CFG["n_sc"], CFG["n_sym"]; KL = K * L
T_SYM = 1e-3 / L                                    # symbol duration incl. CP (1 ms slot)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("Device:", DEV, "| grid", K, "x", L)

## 1. Channel model (3GPP TS 36.104 EPA / EVA / ETU, Jakes Doppler)
$H[k,l]=\sum_p g_p(t_l)\,e^{-j2\pi k\Delta f\tau_p}$, each tap $g_p(t)$ is a sum of 16 sinusoids with random angles of arrival.
ICI is neglected ($f_D/\Delta f \le 6.7\,\%$), a standard assumption stated in the paper.

In [ ]:
PROFILES = {   # delays [ns], powers [dB]
    "EPA": ([0, 30, 70, 90, 110, 190, 410], [0.0, -1.0, -2.0, -3.0, -8.0, -17.2, -20.8]),
    "EVA": ([0, 30, 150, 310, 370, 710, 1090, 1730, 2510], [0.0, -1.5, -1.4, -3.6, -0.6, -9.1, -7.0, -12.0, -16.9]),
    "ETU": ([0, 50, 120, 200, 230, 500, 1600, 2300, 5000], [-1.0, -1.0, -1.0, 0.0, 0.0, 0.0, -3.0, -5.0, -7.0]),
}
# 3GPP TR 38.901 Table 7.7.2-1 (TDL-A) and 7.7.2-3 (TDL-C): normalised delays, powers [dB]; tau_n = tau_norm * DS_desired (Eq. 7.7-1)
TDL_A = ([0.0000, 0.3819, 0.4025, 0.5868, 0.4610, 0.5375, 0.6708, 0.5750, 0.7618, 1.5375, 1.8978, 2.2242, 2.1718, 2.4942, 2.5119,
          3.0582, 4.0810, 4.4579, 4.5695, 4.7966, 5.0066, 5.3043, 9.6586],
         [-13.4, 0.0, -2.2, -4.0, -6.0, -8.2, -9.9, -10.5, -7.5, -15.9, -6.6, -16.7, -12.4, -15.2, -10.8, -11.3, -12.7, -16.2, -18.3,
          -18.9, -16.6, -19.9, -29.7])
TDL_C = ([0.0, 0.2099, 0.2219, 0.2329, 0.2176, 0.6366, 0.6448, 0.6560, 0.6584, 0.7935, 0.8213, 0.9336, 1.2285, 1.3083, 2.1704, 2.7105,
          4.2589, 4.6003, 5.4902, 5.6077, 6.3065, 6.6374, 7.0427, 8.6523],
         [-4.4, -1.2, -3.5, -5.2, -2.5, 0.0, -2.2, -3.9, -7.4, -7.1, -10.7, -11.1, -5.1, -6.8, -8.7, -13.2, -13.9, -13.9, -15.8, -17.1,
          -16.0, -15.7, -21.6, -22.8])
for _n, (_d, _p) in {"TDL-A": TDL_A, "TDL-C": TDL_C}.items():
    PROFILES[_n] = ([x * CFG["tdl_ds_ns"] for x in _d], _p)
K_IDX = torch.arange(K, device=DEV, dtype=torch.float32)
T_IDX = torch.arange(L, device=DEV, dtype=torch.float32) * T_SYM

def gen_channel(B, profile, fd, n_sin=16, gen=None, dscale=None):
    """Returns H: (B, K, L) complex64 with E|H|^2 = 1. fd: tensor (B,) Doppler in Hz.
    dscale: optional tensor (B,) that stretches the profile's delays (delay-spread randomisation, as in 3GPP TR 38.901 TDL scaling)."""
    tau, pdb = PROFILES[profile]
    tau = torch.tensor(tau, device=DEV, dtype=torch.float32) * 1e-9
    tau = tau.expand(B, -1) * (dscale.view(B, 1) if dscale is not None else 1.0)          # (B,P)
    pw = 10 ** (torch.tensor(pdb, device=DEV) / 10); pw = pw / pw.sum()
    P = tau.shape[-1]
    theta = torch.rand(B, P, n_sin, device=DEV, generator=gen) * 2 * math.pi
    phi   = torch.rand(B, P, n_sin, device=DEV, generator=gen) * 2 * math.pi
    phase = 2 * math.pi * fd.view(B, 1, 1, 1) * torch.cos(theta)[..., None] * T_IDX.view(1, 1, 1, L) + phi[..., None]
    g = torch.polar(torch.ones_like(phase), phase).sum(2) * torch.sqrt(pw / n_sin).view(1, P, 1)   # (B,P,L)
    Fm = torch.polar(torch.ones(B, P, K, device=DEV), -2 * math.pi * CFG["scs"] * tau[..., None] * K_IDX.view(1, 1, K))  # (B,P,K)
    return torch.einsum("bpl,bpk->bkl", g, Fm)

## 2. Pilot patterns
**Training family:** random comb patterns (pilot-symbol period 3–7, sub-carrier period 2/3/4/6, random offsets, optional staggering).
**Test patterns:** LTE-CRS-like (irregular symbols), NR-DMRS-like (symbols {2,11}, every 2nd sub-carrier, period 9 → unseen),
Sparse (sub-carrier period 8 → unseen), Block-type (full pilot symbols → unseen pilot *type*).

In [ ]:
def comb_mask(syms, kp, offsets):
    m = np.zeros((K, L), bool)
    for i, l in enumerate(syms): m[offsets[i % len(offsets)]::kp, l] = True
    return m

TEST_PATTERNS = {
    "LTE-CRS":   comb_mask([0, 4, 7, 11], 6, [0, 3]),
    "NR-DMRS":   comb_mask([2, 11], 2, [0]),
    "Sparse":    comb_mask([3, 10], 8, [0, 4]),
    "Block":     comb_mask([0, 7], 1, [0]),
}

def random_train_pattern(rng):
    lp = int(rng.choice([3, 4, 5, 6, 7])); kp = int(rng.choice([2, 3, 4, 6]))
    l0 = int(rng.integers(0, lp)); k0 = int(rng.integers(0, kp))
    offs = [k0, (k0 + kp // 2) % kp] if rng.random() < 0.5 else [k0]
    return comb_mask(list(range(l0, L, lp)), kp, offs)

GRID = np.stack(np.meshgrid(np.arange(K), np.arange(L), indexing="ij"), -1).reshape(-1, 2).astype(float)   # (KL,2) row-major
_PCACHE = {}
def pattern_tensors(mask):
    """Pilot flat indices, 2-D linear-interpolation matrix A (KL x P, nearest-neighbour extrapolation), normalised coords."""
    key = mask.tobytes()
    if key in _PCACHE: return _PCACHE[key]
    idx = np.flatnonzero(mask.ravel()); pts = GRID[idx]; P = len(idx)
    if len(np.unique(pts[:, 1])) == 1 or len(np.unique(pts[:, 0])) == 1:
        A = np.full((KL, P), np.nan)
    else:
        A = LinearNDInterpolator(pts, np.eye(P))(GRID)
    bad = np.isnan(A[:, 0])
    if bad.any():
        nn_ = cKDTree(pts).query(GRID[bad])[1]; A[bad] = 0.0; A[np.flatnonzero(bad), nn_] = 1.0
    coords = GRID / np.array([K - 1, L - 1])
    out = dict(idx=torch.tensor(idx, device=DEV), A=torch.tensor(A, dtype=torch.complex64, device=DEV),
               cp=torch.tensor(coords[idx], dtype=torch.float32, device=DEV), P=P)
    dq = (GRID[:, None, :] - pts[None, :, :]) ** 2                     # squared RE distances (KL,P,2)
    dp = (pts[:, None, :] - pts[None, :, :]) ** 2                      # (P,P,2)
    out["dqp"] = torch.tensor(dq, dtype=torch.float32, device=DEV)
    out["dpp"] = torch.tensor(dp, dtype=torch.float32, device=DEV)
    _PCACHE[key] = out; return out

CQ = torch.tensor(GRID / np.array([K - 1, L - 1]), dtype=torch.float32, device=DEV)   # query coords (KL,2)

def received_pilots(H, pt, snr_db, gen=None):
    """Pilots x=1. Returns LS estimates at pilots (B,P) and noise variance (B,)."""
    B = H.shape[0]; s2 = 10 ** (-snr_db / 10)
    hp = H.reshape(B, -1)[:, pt["idx"]]
    n = torch.complex(torch.randn(hp.shape, device=DEV, generator=gen), torch.randn(hp.shape, device=DEV, generator=gen))
    return hp + n * torch.sqrt(s2 / 2).view(B, 1), s2

for n_, m in TEST_PATTERNS.items():
    print(f"{n_:8s}: {m.sum():3d} pilots ({100 * m.mean():.1f}% overhead)")

## 3. Models
* **Robust-LMMSE filter bank** — for a pilot set $\mathcal P$ and design $(\tau_{max}, f_{D,b})$: $\mathbf R_b=\mathbf R_f(\tau_{max})\otimes\mathbf R_t(f_{D,b})$
  (uniform PDP ⊗ Jakes). With $\mathbf R_{pp}=\mathbf U\Lambda\mathbf U^H$ precomputed per pattern, the estimate for any per-sample noise variance is
  $\hat{\mathbf h}_b=\mathbf R_{hp}\mathbf U(\Lambda+\sigma^2\mathbf I)^{-1}\mathbf U^H\hat{\mathbf h}_{LS}$ — exact, batched, no statistics of the true channel needed.
* **PilotFormer-R (proposed)** — pilot tokens = [Re, Im of LS] + Fourier(k, l) + SNR embedding; 4-layer Transformer encoder over the pilot set;
  RE queries = Fourier(k, l) + light conv embedding of [interpolation, bank estimates] + SNR embedding; 3 cross-attention decoder blocks;
  all attention layers carry a **learnable per-head anisotropic Gaussian locality bias**.
  Output: $\hat H(q)=\sum_b \mathrm{softmax}(\mathbf w^\top \mathbf z_q)_b\,\hat H_b(q) + \Delta(q)$, heads zero-initialised (training starts at the bank average).
* **CNN-R** — ReEsNet-style residual CNN with the *same* inputs, SNR and output head (fair baseline for the Transformer).

In [ ]:
class Fourier(nn.Module):
    def __init__(self, n_freq=10, d=128):
        super().__init__()
        self.register_buffer("f", (2.0 ** torch.arange(n_freq)) * math.pi)
        self.proj = nn.Sequential(nn.Linear(4 * n_freq + 2, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, c):                                   # c: (N,2) in [0,1]
        a = c[..., None] * self.f                           # (N,2,F)
        return self.proj(torch.cat([c, torch.sin(a).flatten(-2), torch.cos(a).flatten(-2)], -1))

class LocalMHA(nn.Module):
    """Multi-head attention with a learnable, per-head anisotropic Gaussian locality bias in the time-frequency plane:
    bias_h(q,p) = -(a_h * dk^2 + b_h * dl^2), dk/dl = sub-carrier / symbol distance between query and pilot.
    Heads are initialised from very local to almost global, and learn their own receptive field."""
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.q, self.k, self.v, self.o = (nn.Linear(d, d) for _ in range(4))
        init = torch.tensor([[0.05, 0.5], [0.01, 0.1], [0.002, 0.02], [1e-5, 1e-4]])[:h] if h <= 4 else torch.full((h, 2), 0.01)
        self.raw = nn.Parameter(torch.log(torch.expm1(init)))        # softplus^-1
    def forward(self, xq, xkv, dist2, need_w=False):
        B, Q, _ = xq.shape; P = xkv.shape[1]
        q = self.q(xq).view(B, Q, self.h, self.dh).transpose(1, 2)
        k = self.k(xkv).view(B, P, self.h, self.dh).transpose(1, 2)
        v = self.v(xkv).view(B, P, self.h, self.dh).transpose(1, 2)
        bias = -(dist2 @ F.softplus(self.raw).T).permute(2, 0, 1)[None].to(q.dtype)   # (1,h,Q,P)
        if need_w:
            w = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(self.dh) + bias, -1)
            out = w @ v; w = w.mean(1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=bias); w = None
        return self.o(out.transpose(1, 2).reshape(B, Q, -1)), w

class Block(nn.Module):
    def __init__(self, d, h, mlp_ratio=4):
        super().__init__()
        self.nq, self.nk, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.att = LocalMHA(d, h)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(), nn.Linear(mlp_ratio * d, d))
    def forward(self, q, mem, dist2, need_w=False):
        kv = self.nk(mem) if mem is not None else None
        a, w = self.att(self.nq(q), kv if kv is not None else self.nq(q), dist2, need_w)
        q = q + a
        return q + self.mlp(self.n2(q)), w

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__(); self.b = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.GELU(), nn.Conv2d(c, c, 3, padding=1))
    def forward(self, x): return x + self.b(x)

from scipy.special import j0 as _j0
def robust_R_np(tau_max, fd_max):
    dk = np.arange(K)[:, None] - np.arange(K)[None, :]; x = 2 * np.pi * dk * CFG["scs"] * tau_max
    Rf = np.where(dk == 0, 1.0 + 0j, (1 - np.exp(-1j * x)) / (1j * np.where(dk == 0, 1, x)))
    dl = np.arange(L)[:, None] - np.arange(L)[None, :]; Rt = _j0(2 * np.pi * fd_max * dl * T_SYM)
    return np.kron(Rf, Rt)
BANK_FD = CFG["bank_fd"]
TAU_MAX = max(PROFILES[p][0][-1] for p in CFG["train_prof"]) * 1e-9 * CFG["train_dscale"][1]
ROBUST_NP = [robust_R_np(TAU_MAX, f) for f in BANK_FD]
NB = len(BANK_FD)
_BCACHE = {}
def bank_tensors(pt):
    key = id(pt)
    if key in _BCACHE: return _BCACHE[key]
    idx = pt["idx"].cpu().numpy(); As, UHs, lams = [], [], []
    for R in ROBUST_NP:
        lam, U = np.linalg.eigh(R[np.ix_(idx, idx)])
        As.append(R[:, idx] @ U); UHs.append(U.conj().T); lams.append(np.clip(lam, 0, None))
    b = dict(A=torch.tensor(np.stack(As), dtype=torch.complex64, device=DEV),
             UH=torch.tensor(np.stack(UHs), dtype=torch.complex64, device=DEV),
             lam=torch.tensor(np.stack(lams), dtype=torch.float32, device=DEV))
    _BCACHE[key] = b; return b
def bank_est(h_ls, pt, s2):
    """Robust-LMMSE filter bank, batched over per-sample noise variance s2 (B,). Returns (B, NB, KL)."""
    b = bank_tensors(pt)
    with torch.autocast(DEV.type, enabled=False):
        z = torch.einsum("bp,nqp->bnq", h_ls.to(torch.complex64), b["UH"])
        z = z / (b["lam"][None] + s2.view(-1, 1, 1))
        return torch.einsum("bnp,nkp->bnk", z, b["A"])

class SNREmb(nn.Module):
    def __init__(self, d):
        super().__init__(); self.net = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, s2): return self.net((torch.log10(s2) / 3.0).view(-1, 1).float())

class PilotFormerR(nn.Module):
    """PilotFormer with (i) SNR conditioning, (ii) robust-LMMSE filter-bank prior: output = per-RE learned mixture of the
    bank estimates + learned residual, (iii) light stem (3x3 conv to 32 ch, then 1x1)."""
    def __init__(self, d=128, heads=4, enc_layers=4, dec_layers=3, use_prior=True, use_snr=True, dec_mlp=None):
        super().__init__()
        self.use_prior, self.use_snr = use_prior, use_snr
        self.pos = Fourier(10, d); self.pin = nn.Linear(2, d); self.snr = SNREmb(d)
        cin = 2 + (2 * NB if use_prior else 0)
        self.stem = nn.Sequential(nn.Conv2d(cin, 32, 3, padding=1), nn.GELU(), nn.Conv2d(32, d, 1))
        self.enc = nn.ModuleList([Block(d, heads) for _ in range(enc_layers)])
        self.dec = nn.ModuleList([Block(d, heads, dec_mlp or CFG["dec_mlp"]) for _ in range(dec_layers)])
        self.norm = nn.LayerNorm(d)
        self.res = nn.Linear(d, 2); nn.init.zeros_(self.res.weight); nn.init.zeros_(self.res.bias)
        if use_prior:
            self.mix = nn.Linear(d, NB); nn.init.zeros_(self.mix.weight); nn.init.zeros_(self.mix.bias)
    def forward(self, h_ls, pt, s2, return_attn=False):
        B = h_ls.shape[0]
        with torch.autocast(DEV.type, enabled=False):
            hi = h_ls @ pt["A"].T
            hb = bank_est(h_ls, pt, s2) if self.use_prior else None
        e = self.snr(s2)[:, None] if self.use_snr else 0.0
        x = self.pin(torch.view_as_real(h_ls)) + self.pos(pt["cp"])[None] + e
        for blk in self.enc: x, _ = blk(x, None, pt["dpp"])
        g = [hi[:, None]] + ([hb] if self.use_prior else [])
        g = torch.view_as_real(torch.cat(g, 1).view(B, -1, K, L)).permute(0, 1, 4, 2, 3).reshape(B, -1, K, L)
        q = self.pos(CQ)[None] + self.stem(g).permute(0, 2, 3, 1).reshape(B, KL, -1) + e
        w = None
        for i, blk in enumerate(self.dec):
            q, w = blk(q, x, pt["dqp"], need_w=return_attn and i == len(self.dec) - 1)
        q = self.norm(q)
        r = torch.view_as_complex(self.res(q).float().contiguous())
        if self.use_prior:
            a = torch.softmax(self.mix(q).float(), -1)                       # (B,KL,NB)
            H = (a.transpose(1, 2) * hb).sum(1) + r
        else:
            H = hi + r
        return (H, w) if return_attn else H

class CNNR(nn.Module):
    """CNN baseline given exactly the same inputs/prior/SNR and the same output head as PilotFormerR (fair comparison)."""
    def __init__(self, c=64, n_blocks=8):
        super().__init__()
        self.inp = nn.Conv2d(2 + 2 * NB + 1, c, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(c) for _ in range(n_blocks)])
        self.out = nn.Conv2d(c, 2 + NB, 3, padding=1); nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)
    def forward(self, h_ls, pt, s2):
        B = h_ls.shape[0]
        with torch.autocast(DEV.type, enabled=False):
            hi = h_ls @ pt["A"].T; hb = bank_est(h_ls, pt, s2)
        g = torch.view_as_real(torch.cat([hi[:, None], hb], 1).view(B, -1, K, L)).permute(0, 1, 4, 2, 3).reshape(B, -1, K, L)
        sp = (torch.log10(s2) / 3.0).view(B, 1, 1, 1).expand(B, 1, K, L).to(g.dtype)
        o = self.out(self.body(self.inp(torch.cat([g, sp], 1)))).float()      # (B, 2+NB, K, L)
        r = torch.complex(o[:, 0], o[:, 1]).reshape(B, KL)
        a = torch.softmax(o[:, 2:].reshape(B, NB, KL), 1)
        return (a * hb).sum(1) + r

class ChannelNet(nn.Module):
    """ChannelNet (Soltani et al., 2019): SRCNN super-resolution + DnCNN denoiser on the interpolated pilot grid.
    Adaptation: real and imaginary parts are processed jointly as 2 channels (the paper used two 1-channel networks)."""
    def __init__(self, depth=20, c=64):
        super().__init__()
        self.sr = nn.Sequential(nn.Conv2d(2, 64, 9, padding=4), nn.ReLU(), nn.Conv2d(64, 32, 1), nn.ReLU(), nn.Conv2d(32, 2, 5, padding=2))
        layers = [nn.Conv2d(2, c, 3, padding=1), nn.ReLU()]
        for _ in range(depth - 2): layers += [nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU()]
        layers += [nn.Conv2d(c, 2, 3, padding=1)]
        self.dn = nn.Sequential(*layers)                      # DnCNN predicts the residual (noise)
    def forward(self, h_ls, pt, s2=None):
        B = h_ls.shape[0]
        with torch.autocast(DEV.type, enabled=False):
            hi = h_ls @ pt["A"].T
        x = torch.view_as_real(hi.view(B, K, L)).permute(0, 3, 1, 2)
        y = self.sr(x); y = y - self.dn(y)
        y = y.float().permute(0, 2, 3, 1).contiguous()
        return torch.view_as_complex(y).reshape(B, KL)

class Channelformer(nn.Module):
    """Channelformer (Luan & Thompson, 2023), offline version: FC(72->216) -> 2-head attention -> FC -> Add&Norm -> conv pre-network
    -> Add&Norm; decoder: conv(12@5x5) + 3 residual blocks + FC upsampling (72 -> 14*72) + conv(1@5x5). Fixed to 72 pilots."""
    def __init__(self, n_in=72, n_out=KL, heads=2, nf=12, kblocks=3, ks=5):
        super().__init__()
        self.n_in, self.heads = n_in, heads
        self.fc_in = nn.Linear(n_in, 3 * n_in); self.fc_o = nn.Linear(n_in, n_in)
        self.ln1 = nn.LayerNorm([n_in, 2]); self.ln2 = nn.LayerNorm([n_in, 2])
        self.pre = nn.Sequential(nn.Conv2d(1, 5, 2, padding="same"), nn.GELU(), nn.Conv2d(5, 1, 2, padding="same"))
        self.c0 = nn.Conv2d(1, nf, ks, padding="same")
        self.blocks = nn.ModuleList([nn.Sequential(nn.Conv2d(nf, nf, ks, padding="same"), nn.ReLU(), nn.Conv2d(nf, nf, ks, padding="same"))
                                     for _ in range(kblocks)])
        self.norms = nn.ModuleList([nn.GroupNorm(1, nf) for _ in range(kblocks)])
        self.up = nn.Linear(n_in, n_out); self.cout = nn.Conv2d(nf, 1, ks, padding="same")
    def forward(self, h_ls, pt, s2=None):
        assert h_ls.shape[1] == self.n_in, "Channelformer is fixed to its training pilot pattern"
        B = h_ls.shape[0]
        x = torch.view_as_real(h_ls).float()                                   # (B,72,2)
        qkv = self.fc_in(x.transpose(1, 2)).transpose(1, 2)                    # (B,216,2)
        q, k, v = qkv.split(self.n_in, 1)
        dh = self.n_in // self.heads
        q, k, v = (t.reshape(B, self.heads, dh, 2) for t in (q, k, v))
        a = torch.softmax(q.transpose(-1, -2) @ k / math.sqrt(dh), -1)        # (B,h,2,2)
        o = (v @ a.transpose(-1, -2)).reshape(B, self.n_in, 2)
        x = self.ln1(x + self.fc_o(o.transpose(1, 2)).transpose(1, 2))
        x = self.ln2(x + self.pre(x[:, None]).squeeze(1))                     # (B,72,2)
        y = self.c0(x[:, None])                                                # (B,nf,72,2)
        for blk, nrm in zip(self.blocks, self.norms): y = nrm(y + blk(y))
        y = self.up(y.transpose(2, 3)).transpose(2, 3)                         # (B,nf,1008,2)
        y = self.cout(y).squeeze(1).float().contiguous()                       # (B,1008,2)
        return torch.view_as_complex(y)

ONLY_PATTERN = {"Channelformer": "NR-DMRS"}      # methods that exist only for one pilot layout

## 4. Training on randomised channels, SNRs, Doppler and pilot patterns (fresh data every step)

In [ ]:
def nmse_loss(Hh, H):
    """Mean of per-slot NMSE in log domain, so high-SNR slots are not drowned out by low-SNR ones."""
    e = ((Hh - H).abs() ** 2).sum(1).div((H.abs() ** 2).sum(1))
    return torch.log10(e + 1e-8).mean(), e.mean()

def find_ckpt(fname):
    p = os.path.join(CFG["out_dir"], fname)
    if os.path.exists(p): return p
    hits = sorted(glob.glob(os.path.join(CFG["ckpt_search"], "**", fname), recursive=True))
    return hits[0] if hits else None

def train(model, name, fixed_pattern=None, steps=None, seed=0, use_dscale=True, lr=None):
    steps = steps or CFG["train_steps"]; rng = np.random.default_rng(seed); lr = lr or CFG["lr"]
    ckpt = os.path.join(CFG["out_dir"], f"{name}_s{seed}.pt")
    found = find_ckpt(f"{name}_s{seed}.pt")
    if found:                                                   # resume / reuse a previous run
        model.load_state_dict(torch.load(found, map_location=DEV)); print(f"[{name} s{seed}] loaded checkpoint {found}")
        if found != ckpt: torch.save(model.state_dict(), ckpt)
        return model.to(DEV).eval()
    torch.manual_seed(seed)
    model.to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=max(0.05, 3 / steps))
    scaler = torch.amp.GradScaler(enabled=AMP); B = CFG["batch"]; t0 = time.time(); hist = []
    for it in range(1, steps + 1):
        prof = CFG["train_prof"][it % len(CFG["train_prof"])]
        mask = fixed_pattern if fixed_pattern is not None else random_train_pattern(rng)
        pt = pattern_tensors(mask)
        with torch.no_grad():
            fd = torch.empty(B, device=DEV).uniform_(*CFG["train_fd"])
            snr = torch.empty(B, device=DEV).uniform_(*CFG["train_snr"])
            ds = torch.empty(B, device=DEV).uniform_(*CFG["train_dscale"]) if use_dscale else None
            H = gen_channel(B, prof, fd, dscale=ds).reshape(B, -1)
            h_ls, _ = received_pilots(H.view(B, K, L), pt, snr)
            s2 = 10 ** (-snr / 10)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
            Hh = model(h_ls, pt, s2)
        loss, lin = nmse_loss(Hh.to(torch.complex64), H)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        hist.append(lin.item())
        if it % max(1, steps // 10) == 0:
            print(f"[{name} s{seed}] step {it:6d}/{steps}  NMSE(train) = {10 * np.log10(np.mean(hist[-200:])):6.2f} dB  ({time.time() - t0:5.0f}s)")
    pd.DataFrame({"step": np.arange(1, steps + 1), "nmse": hist}).to_csv(os.path.join(CFG["out_dir"], f"train_curve_{name}_s{seed}.csv"), index=False)
    torch.save(model.state_dict(), ckpt)
    return model.eval()

MODELS = {}      # name -> list of trained models (one per seed)
S0 = CFG["seeds"][0]
MODELS["PilotFormer"] = [train(PilotFormerR(CFG["d_model"]), "PilotFormer", seed=s) for s in CFG["seeds"]]
MODELS["CNN"] = [train(CNNR(), "CNN", seed=s) for s in CFG["seeds"]]
if CFG["ablations"]:
    MODELS["PilotFormer-Lite"] = [train(PilotFormerR(64, enc_layers=2, dec_layers=2), "PilotFormer-Lite", seed=S0)]
    MODELS["PilotFormer-noPrior"] = [train(PilotFormerR(CFG["d_model"], use_prior=False), "PilotFormer-noPrior", seed=S0)]
    MODELS["PilotFormer-fixed"] = [train(PilotFormerR(CFG["d_model"]), "PilotFormer-fixed", fixed_pattern=TEST_PATTERNS["LTE-CRS"], seed=S0)]
if CFG["abl_no_ds"]:
    MODELS["PilotFormer-noDS"] = [train(PilotFormerR(CFG["d_model"]), "PilotFormer-noDS", seed=S0, use_dscale=False)]
# published baselines: same data generator, loss and step budget; learning rates from the papers
MODELS["ChannelNet"] = [train(ChannelNet(), "ChannelNet", seed=s, lr=1e-3) for s in CFG["base_seeds"]]
MODELS["Channelformer"] = [train(Channelformer(), "Channelformer", fixed_pattern=TEST_PATTERNS["NR-DMRS"], seed=s, lr=2e-3)
                           for s in CFG["base_seeds"]]

## 5. LMMSE baselines
$\hat{\mathbf h}=\mathbf R_{h,p}(\mathbf R_{p,p}+\sigma^2\mathbf I)^{-1}\hat{\mathbf h}_{LS}$ with
(i) the covariance of the **training** distribution (practical, same knowledge as the networks),
(ii) **robust LMMSE** (Li, Cimini & Sollenberger, 1998): uniform PDP over $[0,\tau_{max}]$ ⊗ Jakes at $f_{D}$ — the training-range design
     (5.5 µs, 1 kHz) is the main baseline; **"Robust bank (genie)"** picks, for every test point, the best of the 100/400/1000 Hz designs
     with hindsight (an upper bound for any fixed-filter selection rule),
(iii) the **oracle** covariance of each test condition (genie upper benchmark, needs exact statistics).
All SNR-aware methods (LMMSE variants, PilotFormer-R, CNN-R) receive the same SNR; Section 7b tests SNR-estimation errors.

In [ ]:
@torch.no_grad()
def covariance(sampler, n=CFG["n_cov"], bs=1000):
    R = torch.zeros(KL, KL, dtype=torch.complex64, device=DEV)
    for _ in range(n // bs):
        H = sampler(bs).reshape(bs, -1); R += H.T @ H.conj()
    return R / n

def lmmse(h_ls, s2, R, pt):
    idx = pt["idx"]; Rhp, Rpp = R[:, idx], R[idx][:, idx]
    out = torch.empty(h_ls.shape[0], KL, dtype=torch.complex64, device=DEV)
    for s in torch.unique(s2):                                   # s2 constant within a test batch
        M = Rpp + s * torch.eye(len(idx), device=DEV)
        WT = torch.linalg.solve(M.T, Rhp.T)                     # W = R_hp M^{-1}  ->  W^T = M^{-T} R_hp^T
        sel = s2 == s; out[sel] = h_ls[sel] @ WT
    return out

def train_sampler(bs):
    prof = CFG["train_prof"][np.random.randint(len(CFG["train_prof"]))]
    return gen_channel(bs, prof, torch.empty(bs, device=DEV).uniform_(*CFG["train_fd"]),
                       dscale=torch.empty(bs, device=DEV).uniform_(*CFG["train_dscale"]))
R_TRAIN = covariance(train_sampler)

print(f"robust LMMSE bank: tau_max = {TAU_MAX * 1e6:.2f} us, Doppler designs = {BANK_FD} Hz")

## 6. Evaluation: NMSE and 16-QAM BER (ZF equalisation, Gray mapping) vs SNR

In [ ]:
PAM = torch.tensor([-3.0, -1.0, 3.0, 1.0], device=DEV) / math.sqrt(10)   # Gray: 00,01,10,11 -> -3,-1,3,1

def qam16(bits):                       # bits (...,4) -> complex
    i = PAM[bits[..., 0] * 2 + bits[..., 1]]; q = PAM[bits[..., 2] * 2 + bits[..., 3]]
    return torch.complex(i, q)

def pam_demod(x):
    x = x * math.sqrt(10)
    return torch.stack([(x > 0).long(), (x.abs() < 2).long()], -1)

def ber(H, Hh, mask, s2, gen):
    B = H.shape[0]; data = torch.tensor(~mask.ravel(), device=DEV)
    bits = torch.randint(0, 2, (B, KL, 4), device=DEV, generator=gen)
    X = qam16(bits)
    N = torch.complex(torch.randn(B, KL, device=DEV, generator=gen), torch.randn(B, KL, device=DEV, generator=gen)) * math.sqrt(s2 / 2)
    Y = H * X + N
    Xh = Y * Hh.conj() / (Hh.abs() ** 2 + 1e-12)
    bh = torch.cat([pam_demod(Xh.real), pam_demod(Xh.imag)], -1)
    return (bh != bits)[:, data].float().mean().item()

CONDITIONS = {   # name: (profile, Doppler Hz)
    "EVA-200Hz (seen)":             ("EVA", 200.0),
    "EVA-750Hz (high Doppler)":     ("EVA", 750.0),
    "EVA-1200Hz (beyond training)": ("EVA", 1200.0),
    "ETU-200Hz (unseen profile)":   ("ETU", 200.0),
    "TDL-A-200Hz (3GPP 38.901)":    ("TDL-A", 200.0),
    "TDL-C-200Hz (3GPP 38.901)":    ("TDL-C", 200.0),
}

@torch.no_grad()
def estimate_all(h_ls, s2, pt, R_or, s2_vec, s2_given=None, pattern=None):
    """Returns {method: [estimate per seed]} (classical methods have a single entry).
    s2_vec: true noise variance (used by LMMSE oracle / train-stats); s2_given: SNR estimate handed to SNR-aware methods."""
    sg = s2_vec if s2_given is None else s2_given
    hb = bank_est(h_ls, pt, sg)
    out = {"LS+Interp": [h_ls @ pt["A"].T],
           "LMMSE (train stats)": [lmmse(h_ls, sg, R_TRAIN, pt)],
           "Robust LMMSE": [hb[:, -1]]}                                   # training-range design (1 kHz)
    for i, f in enumerate(BANK_FD): out[f"Robust LMMSE ({int(f)} Hz)"] = [hb[:, i]]
    out["LMMSE (oracle)"] = [lmmse(h_ls, s2_vec, R_or, pt)]
    for n_, ms in MODELS.items():
        if n_ in ONLY_PATTERN and ONLY_PATTERN[n_] != pattern: continue
        out[n_] = []
        for m in ms:
            with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
                out[n_].append(torch.cat([m(h_ls[i:i + 250], pt, sg[i:i + 250]).to(torch.complex64)
                                          for i in range(0, len(h_ls), 250)]))
    return out

def add_genie(df, keys):
    """'Robust bank (genie)': for each test point, the best robust design chosen with hindsight (by NMSE)."""
    rb = df[df.method.str.startswith("Robust LMMSE (")]
    best = rb.loc[rb.groupby(keys)["nmse_db"].idxmin()].assign(method="Robust bank (genie)")
    return pd.concat([df, best], ignore_index=True)

ROWS = []
for cname, (prof, fd) in CONDITIONS.items():
    g = torch.Generator(device=DEV).manual_seed(1234)
    R_or = covariance(lambda bs: gen_channel(bs, prof, torch.full((bs,), fd, device=DEV)))
    H = gen_channel(CFG["n_test"], prof, torch.full((CFG["n_test"],), fd, device=DEV), gen=g).reshape(CFG["n_test"], -1)
    for pname, mask in TEST_PATTERNS.items():
        pt = pattern_tensors(mask)
        for snr in CFG["test_snrs"]:
            s2 = 10 ** (-snr / 10); s2v = torch.full((len(H),), s2, device=DEV)
            gn = torch.Generator(device=DEV).manual_seed(snr + 7)
            h_ls, _ = received_pilots(H.view(-1, K, L), pt, torch.full((len(H),), float(snr), device=DEV), gen=gn)
            ests = estimate_all(h_ls, s2, pt, R_or, s2v, pattern=pname)
            ests["Perfect CSI"] = [H]
            for mname, lst in ests.items():
                for si, Hh in enumerate(lst):
                    gb = torch.Generator(device=DEV).manual_seed(snr + 99)      # same bits/noise for every method
                    nm = ((Hh - H).abs() ** 2).sum().item() / (H.abs() ** 2).sum().item()
                    ROWS.append(dict(condition=cname, pattern=pname, pilots=int(mask.sum()), snr=snr, method=mname, seed=si,
                                     nmse_db=10 * np.log10(nm + 1e-12) if mname != "Perfect CSI" else np.nan,
                                     ber=ber(H, Hh, mask, s2, gb)))
    print("done:", cname)

res = add_genie(pd.DataFrame(ROWS), ["condition", "pattern", "snr"]); res.to_csv(os.path.join(CFG["out_dir"], "results_all.csv"), index=False)

## 7. Doppler sweep (EVA, LTE-CRS, SNR = 20 dB) — inside the 0–1000 Hz training range and beyond it (1200, 1400 Hz)

In [ ]:
SWEEP = []
pt = pattern_tensors(TEST_PATTERNS["LTE-CRS"])
for fd in [0, 100, 200, 300, 400, 500, 600, 750, 900, 1000, 1200, 1400]:
    g = torch.Generator(device=DEV).manual_seed(55)
    H = gen_channel(CFG["n_test"], "EVA", torch.full((CFG["n_test"],), float(fd), device=DEV), gen=g).reshape(CFG["n_test"], -1)
    R_or = covariance(lambda bs: gen_channel(bs, "EVA", torch.full((bs,), float(fd), device=DEV)), n=2000)
    s2v = torch.full((len(H),), 0.01, device=DEV)
    h_ls, _ = received_pilots(H.view(-1, K, L), pt, torch.full((len(H),), 20.0, device=DEV), gen=torch.Generator(device=DEV).manual_seed(3))
    for mname, lst in estimate_all(h_ls, 0.01, pt, R_or, s2v, pattern="LTE-CRS").items():
        for si, Hh in enumerate(lst):
            SWEEP.append(dict(fd=fd, method=mname, seed=si,
                              nmse_db=10 * np.log10(((Hh - H).abs() ** 2).sum().item() / (H.abs() ** 2).sum().item())))
sweep = add_genie(pd.DataFrame(SWEEP), ["fd"]); sweep.to_csv(os.path.join(CFG["out_dir"], "doppler_sweep.csv"), index=False)
print(sweep.pivot_table(index="fd", columns="method", values="nmse_db").round(2))

## 7b. Robustness to SNR-estimation error (LTE-CRS, true SNR = 20 dB)
Every SNR-aware method gets $\hat\sigma^2=\sigma^2\cdot10^{-\Delta/10}$, i.e. an SNR estimate that is $\Delta$ dB too high (Δ>0) or too low (Δ<0).

In [ ]:
MIS = []
pt = pattern_tensors(TEST_PATTERNS["LTE-CRS"])
for cname, (prof, fd) in CONDITIONS.items():
    g = torch.Generator(device=DEV).manual_seed(77)
    H = gen_channel(CFG["n_test"], prof, torch.full((CFG["n_test"],), fd, device=DEV), gen=g).reshape(CFG["n_test"], -1)
    R_or = covariance(lambda bs: gen_channel(bs, prof, torch.full((bs,), fd, device=DEV)), n=2000)
    s2v = torch.full((len(H),), 0.01, device=DEV)
    h_ls, _ = received_pilots(H.view(-1, K, L), pt, torch.full((len(H),), 20.0, device=DEV), gen=torch.Generator(device=DEV).manual_seed(5))
    for off in CFG["snr_offsets"]:
        for mname, lst in estimate_all(h_ls, 0.01, pt, R_or, s2v, s2_given=s2v * 10 ** (-off / 10), pattern="LTE-CRS").items():
            for si, Hh in enumerate(lst):
                MIS.append(dict(condition=cname, offset_db=off, method=mname, seed=si,
                                nmse_db=10 * np.log10(((Hh - H).abs() ** 2).sum().item() / (H.abs() ** 2).sum().item())))
mis = pd.DataFrame(MIS); mis.to_csv(os.path.join(CFG["out_dir"], "snr_mismatch.csv"), index=False)
t6 = mis[mis.method.isin(["LMMSE (train stats)", "Robust LMMSE", "CNN", "PilotFormer"])].pivot_table(
    index=["condition", "method"], columns="offset_db", values="nmse_db").round(2)
print("=== Table 6: NMSE (dB) vs SNR-estimation error (dB) ===\n", t6); t6.to_csv(os.path.join(CFG["out_dir"], "table6_snr_mismatch.csv"))

## 8. Tables for the paper

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
main = [m for m in ["LS+Interp", "LMMSE (train stats)", "Robust LMMSE", "Robust bank (genie)", "LMMSE (oracle)", "ChannelNet", "CNN", "PilotFormer"]
        if m in res.method.unique()]

t1 = res[(res.pattern == "LTE-CRS") & res.method.isin(main)].pivot_table(index=["condition", "method"], columns="snr", values="nmse_db").round(2)
print("=== Table 1: NMSE (dB) vs SNR, LTE-CRS pattern ===\n", t1); t1.to_csv(os.path.join(CFG["out_dir"], "table1_nmse_vs_snr.csv"))

t2 = res[(res.snr == 20) & ~res.method.str.startswith("Robust LMMSE (")].pivot_table(index=["condition", "pattern"], columns="method", values="nmse_db").round(2)
print("\n=== Table 2: pilot-pattern generalisation, NMSE (dB) at 20 dB SNR ===\n", t2); t2.to_csv(os.path.join(CFG["out_dir"], "table2_pattern_generalisation.csv"))

t3 = res[(res.pattern == "LTE-CRS")].pivot_table(index=["condition", "method"], columns="snr", values="ber")
print("\n=== Table 3: 16-QAM BER vs SNR, LTE-CRS ===\n", t3.map(lambda v: f"{v:.2e}")); t3.to_csv(os.path.join(CFG["out_dir"], "table3_ber_vs_snr.csv"))

t7 = res[(res.pattern == "NR-DMRS") & res.method.isin(main + ["Channelformer"])].pivot_table(
    index=["condition", "method"], columns="snr", values="nmse_db").round(2)
print("\n=== Table 7: NR-DMRS — comparison with the published baselines (Channelformer is native to this layout) ===\n", t7)
t7.to_csv(os.path.join(CFG["out_dir"], "table7_nrdmrs_published_baselines.csv"))

g5 = res[(res.snr == 20) & res.method.isin(["PilotFormer", "CNN"])].groupby(["condition", "pattern", "method"])["nmse_db"]
t5 = (g5.mean().round(2).astype(str) + " ± " + g5.std().fillna(0).round(2).astype(str)).unstack("method")
print(f"\n=== Table 5: NMSE (dB) at 20 dB, mean ± std over seeds {CFG['seeds']} ===\n", t5)
t5.to_csv(os.path.join(CFG["out_dir"], "table5_seeds_mean_std.csv"))

@torch.no_grad()
def time_it(fn, reps=20):
    for _ in range(3): fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps): fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / reps
from torch.utils.flop_counter import FlopCounterMode
pt = pattern_tensors(TEST_PATTERNS["LTE-CRS"]); NBT = 256
hb = torch.randn(NBT, pt["P"], dtype=torch.complex64, device=DEV); s2b = torch.full((NBT,), 0.01, device=DEV)
def fp16(fn):
    def g():
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP): return fn()
    return g
def mflops(fn):
    with torch.no_grad(), FlopCounterMode(display=False) as fc: fn()
    return fc.get_total_flops() / NBT / 1e6
cx = [dict(method="LMMSE (one pattern/SNR, filter precomputed)", params=0, mflops_per_slot=2 * 4 * KL * pt["P"] / 1e6,
           ms_per_slot=1e3 * time_it(lambda: lmmse(hb, s2b, R_TRAIN, pt)) / NBT),
      dict(method="Robust LMMSE bank (3 filters, any SNR)", params=0, mflops_per_slot=mflops(lambda: bank_est(hb, pt, s2b)),
           ms_per_slot=1e3 * time_it(lambda: bank_est(hb, pt, s2b)) / NBT)]
pt_nr = pattern_tensors(TEST_PATTERNS["NR-DMRS"]); hb_nr = torch.randn(NBT, pt_nr["P"], dtype=torch.complex64, device=DEV)
for n_, ms in MODELS.items():
    m = ms[0]
    f = (lambda: m(hb_nr, pt_nr, s2b)) if n_ in ONLY_PATTERN else (lambda: m(hb, pt, s2b))
    cx.append(dict(method=n_, params=sum(p.numel() for p in m.parameters()), mflops_per_slot=mflops(f),
                   ms_per_slot=1e3 * time_it(fp16(f)) / NBT))
t4 = pd.DataFrame(cx).round(4); print("\n=== Table 4: complexity (batch 256, fp16 on GPU; MFLOPs per 72x14 slot) ===\n", t4); t4.to_csv(os.path.join(CFG["out_dir"], "table4_complexity.csv"), index=False)

## 9. Figures (300 dpi)

In [ ]:
COL = {"LS+Interp": "#eb6834", "LMMSE (train stats)": "#eda100", "Robust LMMSE": "#008300", "Robust bank (genie)": "#e34948",
       "LMMSE (oracle)": "#0b0b0b", "CNN": "#1baf7a", "PilotFormer": "#2a78d6", "PilotFormer-fixed": "#6250d6",
       "PilotFormer-noPrior": "#52514e", "PilotFormer-Lite": "#52514e", "PilotFormer-noDS": "#52514e", "Perfect CSI": "#52514e",
       "ChannelNet": "#e87ba4", "Channelformer": "#e34948"}
STY = {"LMMSE (oracle)": "--", "Perfect CSI": ":", "Robust bank (genie)": "-.", "PilotFormer-Lite": ":", "PilotFormer-noDS": "--"}
def curve(d, x, y):   # mean over seeds
    return d.groupby(x)[y].mean().sort_index()
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e0",
                     "grid.linewidth": 0.6, "font.size": 10, "savefig.dpi": 300, "savefig.bbox": "tight"})

def panel_plot(metric, pattern, methods, fname, ylabel, logy=False):
    conds = list(CONDITIONS); nc = 3; nr = int(np.ceil(len(conds) / nc))
    fig, axes = plt.subplots(nr, nc, figsize=(4.3 * nc, 3.4 * nr), sharey=True, squeeze=False)
    for ax in axes.ravel()[len(conds):]: ax.set_visible(False)
    axes = axes.ravel()
    for ax, c in zip(axes, conds):
        d = res[(res.condition == c) & (res.pattern == pattern)]
        for m in methods:
            s = curve(d[d.method == m], "snr", metric)
            if len(s): ax.plot(s.index, s.values, STY.get(m, "-"), color=COL[m], lw=2, marker="o", ms=4, label=m)
        ax.set_title(c); ax.set_xlabel("SNR (dB)")
        if logy: ax.set_yscale("log")
    for i in range(0, len(conds), nc): axes[i].set_ylabel(ylabel)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=min(len(l), 5), frameon=False, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], fname)); plt.show()

panel_plot("nmse_db", "LTE-CRS", main, "fig1_nmse_vs_snr.png", "NMSE (dB)")
panel_plot("ber", "LTE-CRS", main + ["Perfect CSI"], "fig2_ber_vs_snr.png", "BER (16-QAM)", logy=True)

In [ ]:
# Pilot-pattern generalisation (EVA-200 Hz): one panel per pattern
c0 = list(CONDITIONS)[0]; pats = list(TEST_PATTERNS)
meths_pat = [m for m in main if m != "Robust bank (genie)"] + [m for m in ["PilotFormer-fixed", "Channelformer"] if m in MODELS]
meths = main + [m for m in ["PilotFormer-fixed"] if m in MODELS]
fig, axes = plt.subplots(1, len(pats), figsize=(4 * len(pats), 3.5), sharey=True)
for ax, p in zip(axes, pats):
    d = res[(res.condition == c0) & (res.pattern == p)]
    for m in meths_pat:
        s = curve(d[d.method == m], "snr", "nmse_db")
        if len(s): ax.plot(s.index, s.values, STY.get(m, "-"), color=COL[m], lw=2, marker="s" if m == "Channelformer" else "o",
                           ms=4 if m == "Channelformer" else 3, label=m)
    ax.set_title(f"{p} ({int(TEST_PATTERNS[p].sum())} pilots)"); ax.set_xlabel("SNR (dB)")
axes[0].set_ylabel("NMSE (dB)")
hl = {}
for ax in axes:
    for h_, l_ in zip(*ax.get_legend_handles_labels()): hl.setdefault(l_, h_)
fig.legend(list(hl.values()), list(hl.keys()), loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.16))
fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig3_pattern_generalisation.png")); plt.show()

In [ ]:
# Doppler sweep
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axvspan(*CFG["train_fd"], color="#e6e5e0", alpha=0.6, lw=0)
for m in meths:
    s = curve(sweep[sweep.method == m], "fd", "nmse_db"); ax.plot(s.index, s.values, STY.get(m, "-"), color=COL[m], lw=2, marker="o", ms=4, label=m)
ax.text(np.mean(CFG["train_fd"]), 0.97, "training range", transform=ax.get_xaxis_transform(), ha="center", va="top", color="#52514e")
ax.axvspan(CFG["train_fd"][1], 1450, color="#fde7dd", alpha=0.5, lw=0)
ax.text(1200, 0.97, "beyond training", transform=ax.get_xaxis_transform(), ha="center", va="top", color="#52514e")
ax.set(xlabel="Maximum Doppler shift (Hz)", ylabel="NMSE (dB)", title="EVA, LTE-CRS pilots, SNR = 20 dB", xlim=(-30, 1450))
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4)
fig.savefig(os.path.join(CFG["out_dir"], "fig4_doppler_sweep.png")); plt.show()

In [ ]:
# Pilot patterns + attention map of one query RE (what the decoder looks at)
fig, axes = plt.subplots(1, len(TEST_PATTERNS) + 1, figsize=(3.0 * (len(TEST_PATTERNS) + 1), 3.8))
for ax, (p, m) in zip(axes, TEST_PATTERNS.items()):
    ax.imshow(m.astype(float), aspect="auto", cmap="Blues", vmin=0, vmax=1.3, origin="lower")
    ax.set(title=p, xlabel="OFDM symbol", ylabel="Sub-carrier"); ax.grid(False)
pt = pattern_tensors(TEST_PATTERNS["LTE-CRS"])
g = torch.Generator(device=DEV).manual_seed(9)
Hs = gen_channel(64, "EVA", torch.full((64,), 200.0, device=DEV), gen=g)
h_ls, _ = received_pilots(Hs, pt, torch.full((64,), 20.0, device=DEV), gen=g)
with torch.no_grad():
    _, w = MODELS["PilotFormer"][0](h_ls, pt, torch.full((64,), 0.01, device=DEV), return_attn=True)   # (B, KL, P)
q_k, q_l = 40, 9; qi = q_k * L + q_l
att = np.zeros((K, L)); att.ravel()[pt["idx"].cpu().numpy()] = w[:, qi].float().mean(0).cpu().numpy()
ax = axes[-1]; im = ax.imshow(att, aspect="auto", cmap="Blues", origin="lower")
ax.plot(q_l, q_k, marker="x", color="#eb6834", ms=10, mew=2); ax.grid(False)
ax.set(title=f"Attention of RE (k={q_k}, l={q_l})", xlabel="OFDM symbol"); fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout(); fig.savefig(os.path.join(CFG["out_dir"], "fig5_patterns_attention.png")); plt.show()

## 10. Output files
All results are written to `OUT_DIR` (default `/kaggle/working`) and zipped as `pilotformer_outputs.zip`:
* `results_all.csv`: NMSE and BER of every estimator for every channel, pilot layout, SNR and seed;
* `table1_nmse_vs_snr.csv`, `table2_pattern_generalisation.csv`, `table3_ber_vs_snr.csv`: summary tables;
* `table4_complexity.csv`: parameters, MFLOP per slot and FP16 latency;
* `table5_seeds_mean_std.csv`: mean and standard deviation over seeds;
* `table6_snr_mismatch.csv`, `snr_mismatch.csv`: estimators fed a wrong SNR (offset of -3 to +3 dB);
* `table7_nrdmrs_published_baselines.csv`: ChannelNet and Channelformer against PilotFormer-R on NR-DMRS;
* `doppler_sweep.csv`: NMSE from 0 to 1400 Hz (training covers 0 to 1000 Hz);
* `train_curve_*.csv`, model checkpoints and diagnostic figures.

## 11. Bundle every output into one ZIP (download it from the Output tab)

In [ ]:
meta = dict(config=CFG, device=str(DEV), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            torch=torch.__version__, python=platform.python_version(),
            pilots={k: int(v.sum()) for k, v in TEST_PATTERNS.items()})
with open(os.path.join(CFG["out_dir"], "run_info.json"), "w") as f: json.dump(meta, f, indent=2, default=str)
zip_path = os.path.join(CFG["out_dir"], "pilotformer_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(os.path.join(CFG["out_dir"], "*"))):
        if p.endswith((".csv", ".png", ".json")): z.write(p, arcname=os.path.basename(p))
    print("\n".join(z.namelist()))
print(f"\nZIP ready: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")